In [0]:
row_orders = spark.read.csv("/Volumes/db_project/sales_data/data_volume/bronze/retail_dataset.csv", header=True,inferSchema=True)
display(row_orders)

In [0]:
raw_customers = spark.read.json("/Volumes/db_project/sales_data/data_volume/bronze/customer_dataset.json")
display(raw_customers)

In [0]:
from pyspark.sql.functions import *
spark.conf.set("spark.sql.ansi.enabled", "false")

order_ts = coalesce(
    try_to_timestamp(trim(col("order_date")), lit("dd/MM/yyyy")),
    try_to_timestamp(trim(col("order_date")), lit("dd-MM-yyyy")),
    try_to_timestamp(trim(col("order_date")), lit("dd-MMM-yyyy")),
    try_to_timestamp(trim(col("order_date")), lit("yyyy-MM-dd")),
    try_to_timestamp(trim(col("order_date")), lit("MM-dd-yyyy"))
)

clean_orders = (
    row_orders
    .withColumn("order_id", trim(col("order_id")).cast("long"))
    .withColumn("order_date_ts", order_ts)
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("product_id", lower(trim(col("product_id"))))
    .withColumn("product_name", lower(trim(col("product_name"))))
    .withColumn("category", lower(trim(col("category"))))
    .withColumn("quantity", when(trim(col("quantity")).isNull() | (trim(col("quantity"))==""), lit(0)).otherwise(trim(col("quantity"))))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("quantity", when(col("quantity") <= 0, lit(0)).otherwise(col("quantity")))
    .withColumn("price_clean", translate(trim(col("price")), "$",""))
    .withColumn("price", col("price_clean").cast("double")).drop("price_clean")
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("returned", lower(trim(col("returned"))))
    .withColumn("total_amount", coalesce(col("quantity"), lit(1)) * coalesce(col("price"), lit(0.0)))
)

clean_orders = clean_orders.dropDuplicates(["order_id", "product_id"]).filter(col("order_id").isNotNull() & col("product_id").isNotNull())

clean_orders.write.mode("overwrite").saveAsTable("db_project.sales_data.silver_orders")

#display(clean_orders)

In [0]:
%sql
select * from db_project.sales_data.silver_orders

In [0]:
raw_customers = spark.read.json("/Volumes/db_project/sales_data/data_volume/bronze/customer_dataset.json")
display(raw_customers)

In [0]:
from pyspark.sql.functions import *
spark.conf.set("spark.sql.ansi.enabled", "false")

customer_ts = coalesce(
    try_to_timestamp(trim(col("signup_date")), lit("dd/MM/yyyy")),
    try_to_timestamp(trim(col("signup_date")), lit("dd-MM-yyyy")),
    try_to_timestamp(trim(col("signup_date")), lit("dd-MMM-yyyy")),
    try_to_timestamp(trim(col("signup_date")), lit("yyyy-MM-dd")),
    try_to_timestamp(trim(col("signup_date")), lit("MM-dd-yyyy"))
)

customer_gender = coalesce(
    when(trim(col("gender")).isNull() | (trim(col("gender"))==""), None),
    when(trim(col("gender"))=="m", "Male"),
    when(trim(col("gender"))=="f", "Female"),
    when(trim(col("gender"))=="M", "Male"),
    when(trim(col("gender"))=="F", "Female"),
    when(trim(col("gender"))=="male", "Male"),
    when(trim(col("gender"))=="female", "Female"),
)



clean_customers = (
    raw_customers
    .withColumn("age", when(trim(col("age")).isNull() | (trim(col("age"))==""), None).otherwise(col("age")))
    .withColumn("age", when(col("age").cast("int") < 0,None).otherwise(col("age").cast("int")))
    .withColumn("city", lower(trim(col("city"))))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("gender", customer_gender)
    .withColumn("loyalty_tier", lower(trim(col("loyalty_tier"))))
    .withColumn("signup_date_ts", customer_ts)
)


clean_customers.write.format("delta").mode("overwrite").saveAsTable("db_project.sales_data.silver_customers")



In [0]:
%sql
select * from db_project.sales_data.silver_customers

In [0]:
silver_orders = spark.read.table("db_project.sales_data.silver_orders")
silver_customers = spark.read.table("db_project.sales_data.silver_customers")

ordered_enrished = silver_orders.join(silver_customers, on="customer_id", how="left")

ordered_enrished = ordered_enrished.withColumn("order_date", to_date(col("order_date_ts")))\
                .withColumn("Year", year(col("order_date_ts")))\
                .withColumn("Month", month(col("order_date_ts")))

gold_df = (
  ordered_enrished.groupBy("product_id","product_name","customer_id","category","year","month","order_date","gender","age","city","loyalty_tier")
  .agg(
    sum("total_amount").alias("total_sales"),
    sum(coalesce(col("quantity"), lit(0))).alias("total_quantity"),
    countDistinct("order_id").alias("total_orders"),
    min("price").alias("Min_Price"),
    max("price").alias("Max_Price"),
    avg("price").alias("Avg_Unit_Price"),
    sum(when(col("returned") == "yes",1).otherwise(0)).alias("retuerns_count"),
    sum(when(col("returned") == "yes", col("total_amount")).otherwise(0.0)).alias("returned_amount")
  )
)

gold_df = gold_df.withColumn("avg_orders_value", when(col("total_orders")>0, col("total_sales")/col("total_orders")).otherwise(lit(0.0)))\
.withColumn("avg_Price_Per_Items", when(col("total_quantity")>0, col("total_sales")/col("total_quantity")).otherwise(lit(0.0))).withColumn("refreshed_at", current_timestamp())

#gold_df.write.format("delta").mode("overwrite").saveAsTable("db_project.sales_data.gold_aggregates")
display(gold_df)




In [0]:
%sql
select * from db_project.sales_data.gold_aggregates